In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
import joblib
import os

def train_model():
    print("1. Loading dataset...")
    try:
        df = pd.read_excel(r'C:\Users\nphss\Komalica-Projects\Ml_Internship_Project\data\telecom_churn_data.xlsx')
    except FileNotFoundError:
        print("Dataset not found! Please make sure 'telecom_churn_data.xlsx' is inside your 'data/' folder.")
        return

    # Drop columns that are IDs, administrative, or leaking the answer (like Churn Reason / Churn Score)
    cols_to_drop = ['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Churn Reason', 'Churn Score', 'CLTV', 'Churn Label']
    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

    print("2. Cleaning data...")
    # Fix 'Total Charges': coerce errors to NaN
    if 'Total Charges' in df.columns:
        df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

    # Target variable is 'Churn Value' (1 for churn, 0 for stay)
    if 'Churn Value' in df.columns:
        y = df['Churn Value']
        X = df.drop('Churn Value', axis=1)
    else:
        print("Error: 'Churn Value' column not found in dataset!")
        return

    # Identify numeric and categorical columns
    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

    print(f"Numeric features found: {numeric_features}")
    print(f"Categorical features found: {categorical_features}")

    print("3. Building the preprocessing and training pipeline...")
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])

    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
    ])

    # Split data into Training set (80%) and Testing set (20%)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    print("4. Training the model...")
    model_pipeline.fit(X_train, y_train)

    accuracy = model_pipeline.score(X_test, y_test)
    print(f"Model trained successfully! Test Accuracy: {accuracy * 100:.2f}%")

    print("5. Saving the model to 'models/model.joblib'...")
    os.makedirs('models', exist_ok=True)
    joblib.dump(model_pipeline, 'models/model.joblib')
    print("Done! Model saved successfully.")

if __name__ == '__main__':
    train_model()

1. Loading dataset...
2. Cleaning data...
Numeric features found: ['Tenure Months', 'Monthly Charges', 'Total Charges']
Categorical features found: ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']
3. Building the preprocessing and training pipeline...
4. Training the model...
Model trained successfully! Test Accuracy: 78.50%
5. Saving the model to 'models/model.joblib'...
Done! Model saved successfully.


In [10]:
import pandas as pd

# Load just the first few rows and print column names
df_check = pd.read_excel('data/telecom_churn_data.xlsx')
print("Here are the exact column names in your file:")
print(df_check.columns.tolist())

Here are the exact column names in your file:
['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']
